In [ ]:
import requests
import time
import warnings
from collections import Counter

# Disable insecure request warnings for local API
warnings.filterwarnings("ignore")

In [ ]:
class GameMonitor:
    def __init__(self, poll_interval=1.0):
        self.url = "https://127.0.0.1:2999/liveclientdata/allgamedata"
        self.poll_interval = poll_interval
        
        # State tracking
        self.previous_items = []
        self.previous_level = 0
        self.previous_kda = (0, 0, 0)
        self.previous_game_time = 0
        self.is_first_run = True

    def fetch_data(self):
        try:
            response = requests.get(self.url, verify=False, timeout=2)
            return response.json()
        except requests.exceptions.RequestException:
            return None
        except Exception as e:
            return None

    def process_items(self, current_item_objects):
        # Extract item names from the objects
        current_item_names = [item['displayName'] for item in current_item_objects]
        
        if self.is_first_run:
            self.previous_items = current_item_names
            return

        # Compare counts to handle purchases
        curr_counts = Counter(current_item_names)
        prev_counts = Counter(self.previous_items)
        
        new_items = []
        for item, count in curr_counts.items():
            prev_count = prev_counts.get(item, 0)
            if count > prev_count:
                new_items.extend([item] * (count - prev_count))
                
        if new_items:
            for item in new_items:
                print(f"[{self.format_time()}] 💰 ITEM BOUGHT: {item}")
        
        self.previous_items = current_item_names

    def format_time(self):
        mins = int(self.previous_game_time // 60)
        secs = int(self.previous_game_time % 60)
        return f"{mins:02d}:{secs:02d}"

    def update(self):
        data = self.fetch_data()
        if not data:
            print("Waiting for game / API not ready...", end="\r")
            return

        try:
            game_time = data['gameData']['gameTime']
            self.previous_game_time = game_time
            
            active_player_name = data['activePlayer']['summonerName']
            
            # Find the active player in the allPlayers list to get scores/items
            # matches on sumonnerName (which combines GameName + TagLine usually)
            player_data = next((p for p in data['allPlayers'] if p['summonerName'] == active_player_name), None)
            
            if not player_data:
                # Fallback if names don't match for some reason (e.g. bots practice tool quirk)
                # Try first player or index logic if needed, but 'allPlayers' should contain activePlayer
                return

            scores = player_data['scores']
            items = player_data['items']
            level = player_data['level']

            # --- Check Events ---
            
            # 1. Level Up
            if level > self.previous_level:
                if not self.is_first_run:
                    print(f"[{self.format_time()}] 🆙 LEVEL UP: {level}")
                self.previous_level = level

            # 2. Items
            self.process_items(items)

            # 3. KDA
            k, d, a = scores['kills'], scores['deaths'], scores['assists']
            if (k, d, a) != self.previous_kda:
                if not self.is_first_run:
                    print(f"[{self.format_time()}] ⚔️ KDA UPDATE: {k}/{d}/{a}")
                self.previous_kda = (k, d, a)

            # 4. Status Heartbeat (Optional: every 30s or just overwriting line)
            # print(f"[{self.format_time()}] Status: {k}/{d}/{a} | {level} | {int(data['activePlayer']['currentGold'])}g", end='\r')

            self.is_first_run = False

        except KeyError as e:
            # API structure can change or be partial during loading
            pass

    def start(self):
        print("Starting Game Monitor...")
        print("Make sure League of Legends is running with a match in progress.")
        try:
            while True:
                self.update()
                time.sleep(self.poll_interval)
        except KeyboardInterrupt:
            print("\nStopped Monitoring.")

In [ ]:
monitor = GameMonitor()
monitor.start()